# 中芯国际 (00981.HK) 技术指标计算实验

> 本 Notebook 基于 2025-01-01 至 2026-07-04 的港股日线数据，
> 从零计算 RSI、MACD、布林带、ATR 四项技术指标，展现完整推导过程。

**标的信息**: 中芯国际 00981.HK | 港交所 | 半导体 | 港元  
**数据范围**: 367 个交易日  
**指标清单**: RSI(14) / MACD(12,26,9) / BOLL(20,2) / ATR(14)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# 中文字体
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

COLOR_UP    = '#e74c3c'
COLOR_DOWN  = '#27ae60'
COLOR_FLAT  = '#95a5a6'
COLOR_BLUE  = '#2c3e50'

DATA_FILE  = Path('../data/00981_HK/daily_quotes.csv')
OUTPUT_DIR = Path('../output')
CHARTS_DIR = OUTPUT_DIR / 'charts'

print('ready')

## 数据加载与预处理

**数据来源**: Tushare hk_daily  
**字段**: trade_date / open / high / low / close / pre_close / vol  
**预处理**:  
1. trade_date 转 datetime，设为索引  
2. 按日期升序排列  
3. 检查缺失值：close 不得有 NaN  
4. 检查合理性：high >= low，close > 0

In [ ]:
df = pd.read_csv(DATA_FILE, encoding='utf-8-sig')
df['trade_date'] = pd.to_datetime(df['trade_date'], format='%Y%m%d')
df = df.set_index('trade_date').sort_index()

assert df['close'].isna().sum() == 0
assert (df['high'] >= df['low']).all()

print(f'数据范围: {df.index[0].date()} ~ {df.index[-1].date()}')
print(f'记录数: {len(df)}')
df[['open','high','low','close','vol']].describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['close'], color=COLOR_BLUE, linewidth=1.2, label='Close')
ax.fill_between(df.index, df['low'], df['high'], alpha=0.1, color=COLOR_BLUE)
ax.set_title(f'SMIC (00981.HK) Close Price  {df.index[0].date()} ~ {df.index[-1].date()}')
ax.set_ylabel('Price (HKD)')
ax.set_xlabel('Date')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## RSI — 相对强弱指标

**类别**: 动量类  
**参数**: 14日  
**公式**: RSI = 100 - 100/(1+RS), RS = 平均涨幅/平均跌幅（Wilder平滑）  
**信号**: RSI > 70 超买（警惕回调），RSI < 30 超卖（关注反弹）

In [ ]:
df['price_change'] = df['close'].diff()
df['gain'] = df['price_change'].clip(lower=0)
df['loss'] = (-df['price_change']).clip(lower=0)

period = 14
df['avg_gain'] = df['gain'].ewm(alpha=1/period, adjust=False).mean()
df['avg_loss'] = df['loss'].ewm(alpha=1/period, adjust=False).mean()
df['rs'] = df['avg_gain'] / df['avg_loss']
df['rsi_14'] = 100 - (100 / (1 + df['rs']))
df.loc[df['avg_loss'] == 0, 'rsi_14'] = 100

print(f"RSI range: {df['rsi_14'].min():.2f} ~ {df['rsi_14'].max():.2f}")
print(f"Current RSI: {df['rsi_14'].iloc[-1]:.2f}")
print(f"Overbought days: {(df['rsi_14'] > 70).sum()}")
print(f"Oversold days:  {(df['rsi_14'] < 30).sum()}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7),
                               gridspec_kw={'height_ratios': [3, 2]},
                               sharex=True)
ax1.plot(df.index, df['close'], color=COLOR_BLUE, linewidth=1.2)
ax1.set_title('SMIC (00981.HK) RSI Analysis')
ax1.set_ylabel('Price (HKD)')
ax1.grid(True, alpha=0.3, linestyle='--')

ax2.plot(df.index, df['rsi_14'], color='#8e44ad', linewidth=1.2)
ax2.axhline(y=70, color=COLOR_DOWN, linestyle='--', alpha=0.7, label='Overbought (70)')
ax2.axhline(y=30, color=COLOR_UP, linestyle='--', alpha=0.7, label='Oversold (30)')
ax2.axhline(y=50, color=COLOR_FLAT, linestyle=':', alpha=0.5)
ax2.fill_between(df.index, 70, 100, alpha=0.08, color=COLOR_DOWN)
ax2.fill_between(df.index, 0, 30, alpha=0.08, color=COLOR_UP)
ax2.set_ylabel('RSI')
ax2.set_xlabel('Date')
ax2.set_ylim(0, 100)
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.legend(loc='upper left', fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## MACD — 指数平滑异同移动平均线

**类别**: 趋势类  
**参数**: 快线 12日，慢线 26日，信号线 9日  
**公式**: DIF = EMA12 - EMA26，DEA = EMA(DIF, 9)，MACD柱状图 = (DIF - DEA) * 2  
**信号**: DIF 上穿 DEA = 金叉（买入），DIF 下穿 DEA = 死叉（卖出）

In [ ]:
def ema(s, period):
    return s.ewm(span=period, adjust=False).mean()

print('EMA helper ready')

In [ ]:
df['ema_12'] = ema(df['close'], 12)
df['ema_26'] = ema(df['close'], 26)
df['dif']    = df['ema_12'] - df['ema_26']
df['dea']    = ema(df['dif'], 9)
df['macd_hist'] = (df['dif'] - df['dea']) * 2

# Count crosses
df['dif_above'] = df['dif'] > df['dea']
df['cross'] = df['dif_above'].diff()
golden = df[df['cross'] == True]
death  = df[df['cross'] == False]

print(f'Golden crosses: {len(golden)}')
print(f'Death  crosses: {len(death)}')
if len(golden) > 0: print(f'Last golden: {golden.index[-1].date()}')
if len(death) > 0:  print(f'Last death:  {death.index[-1].date()}')
df[['close','ema_12','ema_26','dif','dea','macd_hist']].tail(10).round(3)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 9),
                                    gridspec_kw={'height_ratios': [3, 2, 1.5]},
                                    sharex=True)

# Price + EMA
ax1.plot(df.index, df['close'],    color=COLOR_BLUE, lw=1,   label='Close')
ax1.plot(df.index, df['ema_12'],   color=COLOR_UP,   lw=1,   label='EMA12')
ax1.plot(df.index, df['ema_26'],   color=COLOR_DOWN, lw=1,   label='EMA26')
ax1.set_title('SMIC (00981.HK) MACD Analysis')
ax1.set_ylabel('Price (HKD)')
ax1.grid(True, alpha=0.3, ls='--')
ax1.legend(loc='upper left', fontsize=9)

# DIF + DEA
ax2.plot(df.index, df['dif'], color=COLOR_UP,   lw=1, label='DIF')
ax2.plot(df.index, df['dea'], color=COLOR_DOWN, lw=1, label='DEA')
ax2.axhline(y=0, color=COLOR_FLAT, lw=0.5)
ax2.set_ylabel('DIF / DEA')
ax2.grid(True, alpha=0.3, ls='--')
ax2.legend(loc='upper left', fontsize=9)

# MACD Histogram
colors = [COLOR_UP if v >= 0 else COLOR_DOWN for v in df['macd_hist']]
ax3.bar(df.index, df['macd_hist'], color=colors, width=1, alpha=0.7)
ax3.axhline(y=0, color=COLOR_FLAT, lw=0.5)
ax3.set_ylabel('MACD Hist')
ax3.set_xlabel('Date')
ax3.grid(True, alpha=0.3, ls='--')

ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 布林带 (Bollinger Bands)

**类别**: 统计类  
**参数**: 周期 20日，标准差倍数 2  
**公式**: 中轨 = SMA20，上轨 = 中轨 + 2*sigma，下轨 = 中轨 - 2*sigma  
%B = (Close - 下轨) / (上轨 - 下轨)，带宽 = (上轨 - 下轨) / 中轨 * 100%  
**信号**: 价格触及上轨警惕回落，触及下轨关注反弹，带宽收窄可能酝酿突破

In [ ]:
period_bb = 20
std_mult  = 2

df['bb_ma']       = df['close'].rolling(period_bb).mean()
df['bb_std']      = df['close'].rolling(period_bb).std(ddof=0)
df['bb_upper']    = df['bb_ma'] + std_mult * df['bb_std']
df['bb_lower']    = df['bb_ma'] - std_mult * df['bb_std']
df['bb_pct_b']    = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
df['bb_bandwidth'] = (df['bb_upper'] - df['bb_lower']) / df['bb_ma'] * 100

print(f"Current %B:      {df['bb_pct_b'].iloc[-1]:.3f}")
print(f"Current bandwidth: {df['bb_bandwidth'].iloc[-1]:.2f}%")
print(f"Min bandwidth:    {df['bb_bandwidth'].min():.2f}%")
df[['close','bb_ma','bb_upper','bb_lower','bb_pct_b','bb_bandwidth']].tail(10).round(3)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7),
                               gridspec_kw={'height_ratios': [3, 1.5]},
                               sharex=True)

ax1.plot(df.index, df['close'],    color=COLOR_BLUE, lw=1.2, label='Close')
ax1.plot(df.index, df['bb_upper'], color=COLOR_DOWN, lw=0.8, label='Upper (+2s)')
ax1.plot(df.index, df['bb_ma'],    color=COLOR_FLAT, lw=0.8, ls='--', label='MA20')
ax1.plot(df.index, df['bb_lower'], color=COLOR_UP,   lw=0.8, label='Lower (-2s)')
ax1.fill_between(df.index, df['bb_lower'], df['bb_upper'], alpha=0.06, color=COLOR_BLUE)
ax1.set_title('SMIC (00981.HK) Bollinger Bands Analysis')
ax1.set_ylabel('Price (HKD)')
ax1.grid(True, alpha=0.3, ls='--')
ax1.legend(loc='upper left', fontsize=9)

ax2.plot(df.index, df['bb_pct_b'], color='#8e44ad', lw=1, label='%B')
ax2.axhline(y=1.0, color=COLOR_DOWN, ls='--', alpha=0.5, label='Overbought')
ax2.axhline(y=0.0, color=COLOR_UP,   ls='--', alpha=0.5, label='Oversold')
ax2.axhline(y=0.5, color=COLOR_FLAT, ls=':',  alpha=0.4)
ax2.fill_between(df.index, 1.0, 2.0, alpha=0.05, color=COLOR_DOWN)
ax2.fill_between(df.index, 0.0, -0.5, alpha=0.05, color=COLOR_UP)
ax2.set_ylabel('%B')
ax2.set_xlabel('Date')
ax2.set_ylim(-0.5, 2)
ax2.grid(True, alpha=0.3, ls='--')
ax2.legend(loc='upper left', fontsize=9)

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## ATR — 真实波幅指标

**类别**: 波动率类  
**参数**: 14日  
**公式**:  
- TR = max(当日最高-最低, |最高-昨收|, |最低-昨收|)  
- ATR = TR 的 14日 Wilder 平滑  
- ATR% = ATR / Close * 100%  
**用途**: 止损设置（止损 = 买入价 - 2*ATR），仓位管理

In [ ]:
df['h_l']  = df['high'] - df['low']
df['h_pc'] = (df['high'] - df['close'].shift(1)).abs()
df['l_pc'] = (df['low']  - df['close'].shift(1)).abs()
df['tr']   = df[['h_l','h_pc','l_pc']].max(axis=1)
df['atr_14']      = df['tr'].ewm(alpha=1/14, adjust=False).mean()
df['atr_pct']     = df['atr_14'] / df['close'] * 100
df['atr_stop_long']  = df['close'] - 2 * df['atr_14']
df['atr_stop_short'] = df['close'] + 2 * df['atr_14']

print(f"ATR range: {df['atr_14'].min():.2f} ~ {df['atr_14'].max():.2f} HKD")
print(f"Current ATR: {df['atr_14'].iloc[-1]:.2f} HKD ({df['atr_pct'].iloc[-1]:.1f}%)")
df[['close','atr_14','atr_pct','atr_stop_long']].tail(10).round(3)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 9),
                                    gridspec_kw={'height_ratios': [3, 1.5, 1.5]},
                                    sharex=True)

# Price + ATR bands
ax1.plot(df.index, df['close'], color=COLOR_BLUE, lw=1.2, label='Close')
ax1.plot(df.index, df['atr_stop_long'],  color=COLOR_DOWN, lw=0.8, ls=':', label='Stop Loss (Long, -2ATR)')
ax1.plot(df.index, df['atr_stop_short'], color=COLOR_UP,   lw=0.8, ls=':', label='Stop Loss (Short, +2ATR)')
ax1.fill_between(df.index, df['atr_stop_long'], df['atr_stop_short'],
                 alpha=0.05, color=COLOR_BLUE, label='ATR Band')
ax1.set_title('SMIC (00981.HK) ATR Analysis')
ax1.set_ylabel('Price (HKD)')
ax1.grid(True, alpha=0.3, ls='--')
ax1.legend(loc='upper left', fontsize=8)

# ATR line
ax2.plot(df.index, df['atr_14'], color='#2980b9', lw=1.2)
ax2.set_ylabel('ATR (HKD)')
ax2.grid(True, alpha=0.3, ls='--')

# ATR%
ax3.plot(df.index, df['atr_pct'], color='#8e44ad', lw=1.2)
ax3.set_ylabel('ATR %')
ax3.set_xlabel('Date')
ax3.grid(True, alpha=0.3, ls='--')

ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 综合分析

四项指标各管一面：

| 维度 | 指标 | 说明 |
|------|------|------|
| 趋势方向 | MACD | DIF上穿DEA=金叉，下穿=死叉 |
| 动量强度 | RSI | >70超买，<30超卖 |
| 价格位置 | 布林带 | %B>1突破上轨，<0突破下轨 |
| 波动幅度 | ATR | 用于止损设置和仓位管理 |

以下将两个指标放在同一张图中交叉解读（截至 2026-07-03）。

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 12),
                         gridspec_kw={'height_ratios': [2, 1.5, 1.5, 1.5]},
                         sharex=True)

# Panel 1: Price + Bollinger
ax = axes[0]
ax.plot(df.index, df['close'],   color=COLOR_BLUE, lw=1.2, label='Close')
ax.plot(df.index, df['bb_upper'], color=COLOR_DOWN, lw=0.7, label='BB Upper')
ax.plot(df.index, df['bb_ma'],   color=COLOR_FLAT, lw=0.7, ls='--', label='BB MA')
ax.plot(df.index, df['bb_lower'], color=COLOR_UP,   lw=0.7, label='BB Lower')
ax.fill_between(df.index, df['bb_lower'], df['bb_upper'], alpha=0.06, color=COLOR_BLUE)
ax.set_title('SMIC (00981.HK) — 4 Indicators Dashboard', fontsize=14)
ax.set_ylabel('Price (HKD)')
ax.grid(True, alpha=0.3, ls='--')
ax.legend(loc='upper left', fontsize=8)

# Panel 2: MACD
ax = axes[1]
ax.plot(df.index, df['dif'], color=COLOR_UP,   lw=1,   label='DIF')
ax.plot(df.index, df['dea'], color=COLOR_DOWN, lw=1,   label='DEA')
colors = [COLOR_UP if v >= 0 else COLOR_DOWN for v in df['macd_hist']]
ax.bar(df.index, df['macd_hist'], color=colors, width=1, alpha=0.6)
ax.axhline(y=0, color=COLOR_FLAT, lw=0.5)
ax.set_ylabel('MACD')
ax.grid(True, alpha=0.3, ls='--')
ax.legend(loc='upper left', fontsize=8)

# Panel 3: RSI
ax = axes[2]
ax.plot(df.index, df['rsi_14'], color='#8e44ad', lw=1, label='RSI(14)')
ax.axhline(y=70, color=COLOR_DOWN, ls='--', alpha=0.6)
ax.axhline(y=30, color=COLOR_UP,   ls='--', alpha=0.6)
ax.axhline(y=50, color=COLOR_FLAT, ls=':',  alpha=0.4)
ax.set_ylabel('RSI')
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, ls='--')

# Panel 4: ATR
ax = axes[3]
ax.plot(df.index, df['atr_14'], color='#2980b9', lw=1, label='ATR(14)')
ax.set_ylabel('ATR (HKD)')
ax.set_xlabel('Date')
ax.grid(True, alpha=0.3, ls='--')

axes[3].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[3].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
out_cols = [
    'close','rsi_14',
    'dif','dea','macd_hist',
    'bb_ma','bb_upper','bb_lower','bb_pct_b','bb_bandwidth',
    'tr','atr_14','atr_pct','atr_stop_long'
]
out = df[out_cols].copy()
out.to_csv(OUTPUT_DIR / 'smic_00981HK_indicators.csv', encoding='utf-8-sig')
print(f'Exported: {len(out)} rows x {len(out_cols)} cols')
print(f'File: {OUTPUT_DIR / "smic_00981HK_indicators.csv"}')

## 结论与延伸

### 指标解读（截至 2026-07-03）

| 指标 | 当前值 | 信号 |
|------|--------|------|
| RSI(14) | 48.97 | 中性区域 |
| MACD | DIF与DEA接近 | 动量减弱 |
| 布林带 %B | 0.467 | 通道中部 |
| ATR(14) | 6.21 HKD (8.0%) | 波动率中等偏高 |

### 后续方向
1. **回测框架**：基于指标信号构建买卖规则，评估策略收益率
2. **多标的对比**：将同样方法应用于比亚迪(01211.HK)、长江电力(600900.SH)
3. **参数优化**：网格搜索各指标最优参数组合
4. **机器学习**：将四项指标作为特征，训练股价方向预测模型